## 1. Veri Yükleme

In [5]:
import pandas as pd
import numpy as np
import seaborn as sns


In [6]:
# @title
from huggingface_hub import login
login("")

In [7]:
from datasets import load_dataset

ds = load_dataset("manueltonneau/turkish-hate-speech-superset")

In [8]:
df = ds['train'].to_pandas()

## 2. Metin Ön İşleme

In [9]:
df

,text,labels,source,dataset,nb_annotators
0,iyi partili türkkan 1.5 milyon suriyeli daha g...,0,Twitter,mayda_et_al_1,2.0
1,"başkan erdoğan'dan suriyeli mülteci mesajı! ""y...",0,Twitter,mayda_et_al_1,2.0
2,"arslan bulut yazdı ""300 metrekarelik konaklar ...",1,Twitter,mayda_et_al_1,2.0
3,güzel bir amaçla yola çıkmış ama insan askerin...,0,Twitter,mayda_et_al_1,2.0
4,kimin parası bu türk halkının parası peki bizi...,1,Twitter,mayda_et_al_1,2.0
...,...,...,...,...,...
41418,nuray.organizasyon@USER çok teşekkür ederim 🤗b...,0,Instagram,HATC,2.0
41419,emine.sunan12Merhabalar arkadaşlar bu paylaşım...,0,Instagram,HATC,2.0
41420,iremmkrtls 👌🏽👌🏽,0,Instagram,HATC,2.0
41421,jila88.j@USER ay gülmekten karnima ağrilar gır...,0,Instagram,HATC,2.0


In [10]:
import re
def metin_temizle(metin):
    metin = str(metin).lower()
    metin = re.sub(r'http\S+', '', metin)
    metin = re.sub(r'@\w+', '', metin)
    metin = re.sub(r'#\w+', '', metin)

    metin = re.sub(r'[^\w\s\u0080-\uffff]', '', metin)
    metin = re.sub(r'[^\u0000-\u05FF\s]', '', metin)
    metin = re.sub(r'\s+', ' ', metin)
    metin = metin.strip()
    return metin


df["text_clean"]=df["text"].apply(metin_temizle)
print(df[['text', 'text_clean']])

                                                    text  \
0      iyi partili türkkan 1.5 milyon suriyeli daha g...   
1      başkan erdoğan'dan suriyeli mülteci mesajı! "y...   
2      arslan bulut yazdı "300 metrekarelik konaklar ...   
3      güzel bir amaçla yola çıkmış ama insan askerin...   
4      kimin parası bu türk halkının parası peki bizi...   
...                                                  ...   
41418  nuray.organizasyon@USER çok teşekkür ederim 🤗b...   
41419  emine.sunan12Merhabalar arkadaşlar bu paylaşım...   
41420                                    iremmkrtls 👌🏽👌🏽   
41421  jila88.j@USER ay gülmekten karnima ağrilar gır...   
41422  ntkm1234 Çağlada estetik yok botoks var kadın ...   

                                              text_clean  
0      iyi partili türkkan 15 milyon suriyeli daha ge...  
1      başkan erdoğandan suriyeli mülteci mesajı ya d...  
2      arslan bulut yazdı 300 metrekarelik konaklar y...  
3      güzel bir amaçla yola çıkmış ama ins

## 3. Baseline Model: TF-IDF + Logistic Regression

In [11]:
X=df["text_clean"]
y=df["labels"]


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [13]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [14]:
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model=LogisticRegression(max_iter=1000,class_weight="balanced")
model.fit(X_train_tfidf,y_train)

print(classification_report(y_test, model.predict(X_test_tfidf)))

              precision    recall  f1-score   support

           0       0.90      0.92      0.91      5581
           1       0.83      0.79      0.81      2704

    accuracy                           0.88      8285
   macro avg       0.87      0.86      0.86      8285
weighted avg       0.88      0.88      0.88      8285



## 4. Ek Veri Setleri Denemesi (Sonuç: Performansı Düşürdü)

In [15]:
import pandas as pd
!wget -O "Türkçe Nefret Söylemi Veri Seti v1.xlsx" "https://raw.githubusercontent.com/imayda/turkish-hate-speech-dataset-1/main/T%C3%BCrk%C3%A7e%20Nefret%20S%C3%B6ylemi%20Veri%20Seti%20v1.xlsx"
df1=pd.read_excel("Türkçe Nefret Söylemi Veri Seti v1.xlsx", sheet_name=1)

--2026-06-25 14:09:02--  https://raw.githubusercontent.com/imayda/turkish-hate-speech-dataset-1/main/T%C3%BCrk%C3%A7e%20Nefret%20S%C3%B6ylemi%20Veri%20Seti%20v1.xlsx
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 342096 (334K) [application/octet-stream]
Saving to: ‘Türkçe Nefret Söylemi Veri Seti v1.xlsx’

Türkçe Nefret Söyle 100%[===================>] 334.08K  --.-KB/s    in 0.02s   

2026-06-25 14:09:02 (15.8 MB/s) - ‘Türkçe Nefret Söylemi Veri Seti v1.xlsx’ saved [342096/342096]



In [16]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 29 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   row ID                1000 non-null   object 
 1   Tweet                 1000 non-null   object 
 2   Etiket                1000 non-null   object 
 3   Alt Etiket            276 non-null    object 
 4   Tweet ID              1000 non-null   float64
 5   Time                  1000 non-null   object 
 6   Favorited             1000 non-null   int64  
 7   Retweeted             1000 non-null   int64  
 8   Is Favourited         1000 non-null   int64  
 9   Is Retweeted          1000 non-null   int64  
 10  Is Retweet            1000 non-null   int64  
 11  Retweet from          0 non-null      float64
 12  Latitude              4 non-null      float64
 13  Longitude             4 non-null      float64
 14  Country               36 non-null     object 
 15  User                  

In [17]:
df1.head()

,row ID,Tweet,Etiket,Alt Etiket,Tweet ID,Time,Favorited,Retweeted,Is Favourited,Is Retweeted,...,User - Description,User - URL,User - Creation time,User - Language,User - Location,User - Time Zone,User - Statuses,User - Followers,User - Friends,User - Favourites
0,Row0,iyi partili türkkan 1.5 milyon suriyeli daha g...,hiçbiri,NaN,1.169700e+22,2019-09-05 22:36:26,107,46,0,0,...,Sözcü Gazetesi'nin resmi Twitter hesabıdır.,http://t.co/Vw2jdlTPiH,2010-11-21 13:42:48,NaN,İstanbul,NaN,609068,1904801,25,0
1,Row1,"başkan erdoğan'dan suriyeli mülteci mesajı! ""y...",hiçbiri,NaN,1.169550e+23,2019-09-05 13:11:52,282,61,0,0,...,Haberin Kanalı. A Haber'in Resmi Twitter Hesab...,https://t.co/l0318Sc0Kh,2011-04-21 20:05:15,NaN,Türkiye,NaN,484210,1327241,35,2
2,Row2,"arslan bulut yazdı ""300 metrekarelik konaklar ...",nefret,etnik,1.169720e+23,2019-09-06 00:06:00,134,51,0,0,...,Yeniçağ Gazetesi Resmi Hesabıdır - DÜNYAYI TÜR...,http://t.co/HtJfH8NPv6,2013-12-25 18:59:53,NaN,İstanbul,NaN,103506,113321,24,29
3,Row6,güzel bir amaçla yola çıkmış ama insan askerin...,hiçbiri,NaN,1.170000e+18,2019-09-06 15:08:18,1,0,0,0,...,Düşerken tutunduğum tuğlayı kendime rab bellem...,https://t.co/0pvTkdWZzL,2019-06-28 13:34:04,NaN,St Petersburg,NaN,351,77,216,679
4,Row8,kimin parası bu türk halkının parası peki bizi...,nefret,etnik,1.169950e+23,2019-09-06 15:08:01,0,0,0,0,...,"Düşün kadarsin, düşün ne kadarsin? inşaat mühe...",https://t.co/xcx3mCTvwA,2009-04-12 14:11:45,NaN,"İstanbul, Türkiye",NaN,6893,843,1187,15414


In [18]:
df

,text,labels,source,dataset,nb_annotators,text_clean
0,iyi partili türkkan 1.5 milyon suriyeli daha g...,0,Twitter,mayda_et_al_1,2.0,iyi partili türkkan 15 milyon suriyeli daha ge...
1,"başkan erdoğan'dan suriyeli mülteci mesajı! ""y...",0,Twitter,mayda_et_al_1,2.0,başkan erdoğandan suriyeli mülteci mesajı ya d...
2,"arslan bulut yazdı ""300 metrekarelik konaklar ...",1,Twitter,mayda_et_al_1,2.0,arslan bulut yazdı 300 metrekarelik konaklar y...
3,güzel bir amaçla yola çıkmış ama insan askerin...,0,Twitter,mayda_et_al_1,2.0,güzel bir amaçla yola çıkmış ama insan askerin...
4,kimin parası bu türk halkının parası peki bizi...,1,Twitter,mayda_et_al_1,2.0,kimin parası bu türk halkının parası peki bizi...
...,...,...,...,...,...,...
41418,nuray.organizasyon@USER çok teşekkür ederim 🤗b...,0,Instagram,HATC,2.0,nurayorganizasyon çok teşekkür ederim beğenmen...
41419,emine.sunan12Merhabalar arkadaşlar bu paylaşım...,0,Instagram,HATC,2.0,eminesunan12merhabalar arkadaşlar bu paylaşıml...
41420,iremmkrtls 👌🏽👌🏽,0,Instagram,HATC,2.0,iremmkrtls
41421,jila88.j@USER ay gülmekten karnima ağrilar gır...,0,Instagram,HATC,2.0,jila88j ay gülmekten karnima ağrilar gırdı sen...


In [19]:
df1=df1[["Tweet","Etiket"]]

In [20]:
df1.isnull().value_counts()

,,count
Tweet,Etiket,
False,False,1000


In [21]:
df1.loc[df1["Etiket"] == "nefret", "Etiket"] = 1

In [22]:
df1=df1[["Tweet","Etiket"]]

In [23]:
df1["Etiket"].value_counts()

,count
Etiket,
hiçbiri,664
1,276
saldırgan,60


In [24]:
df1.loc[df1["Etiket"] == "saldırgan", "Etiket"] = 1

In [25]:
df1.loc[df1["Etiket"] == "hiçbiri", "Etiket"] = 0

In [26]:
df1["Etiket"].value_counts()

,count
Etiket,
0,664
1,336


In [27]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Tweet   1000 non-null   object
 1   Etiket  1000 non-null   object
dtypes: object(2)
memory usage: 15.8+ KB


In [28]:
df1["Etiket"] = df1["Etiket"].astype(int)

In [29]:
df1["text_clean"]=df1["Tweet"].apply(metin_temizle)
print(df1[['Tweet', 'text_clean']])

                                                 Tweet  \
0    iyi partili türkkan 1.5 milyon suriyeli daha g...   
1    başkan erdoğan'dan suriyeli mülteci mesajı! "y...   
2    arslan bulut yazdı "300 metrekarelik konaklar ...   
3    güzel bir amaçla yola çıkmış ama insan askerin...   
4    kimin parası bu türk halkının parası peki bizi...   
..                                                 ...   
995  bu sadece kadın erkek meselesi mi hayır iyi in...   
996  eşcinsel genin olmadığına dair bilimsel çalışm...   
997  eşcinsel çiftin düğün fotoğrafları viral oldu ...   
998  kral şakir necati karakteri... türk dizilerind...   
999  ulan maçı unutayım kafam dağılsın diye netflix...   

                                            text_clean  
0    iyi partili türkkan 15 milyon suriyeli daha ge...  
1    başkan erdoğandan suriyeli mülteci mesajı ya d...  
2    arslan bulut yazdı 300 metrekarelik konaklar y...  
3    güzel bir amaçla yola çıkmış ama insan askerin...  
4    kimin parası 

In [30]:
df1 = df1.drop(columns=["Tweet"])

In [31]:
df1

,Etiket,text_clean
0,0,iyi partili türkkan 15 milyon suriyeli daha ge...
1,0,başkan erdoğandan suriyeli mülteci mesajı ya d...
2,1,arslan bulut yazdı 300 metrekarelik konaklar y...
3,0,güzel bir amaçla yola çıkmış ama insan askerin...
4,1,kimin parası bu türk halkının parası peki bizi...
...,...,...
995,0,bu sadece kadın erkek meselesi mi hayır iyi in...
996,1,eşcinsel genin olmadığına dair bilimsel çalışm...
997,0,eşcinsel çiftin düğün fotoğrafları viral oldu
998,0,kral şakir necati karakteri türk dizilerinde e...


In [32]:
X=df1["text_clean"]
y=df1["Etiket"]

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model=LogisticRegression(max_iter=1000,class_weight="balanced")
model.fit(X_train_tfidf,y_train)

print(classification_report(y_test, model.predict(X_test_tfidf)))

              precision    recall  f1-score   support

           0       0.80      0.74      0.77       133
           1       0.55      0.63      0.59        67

    accuracy                           0.70       200
   macro avg       0.68      0.69      0.68       200
weighted avg       0.72      0.70      0.71       200



In [33]:
import pandas as pd
!wget -O "9KTurkishHateSpeechDataset.xlsx" "https://raw.githubusercontent.com/mzahidgurbuz/Turkish-Hate-Speech-Detection/main/9KTurkishHateSpeechDataset.xlsx"
df2=pd.read_excel("9KTurkishHateSpeechDataset.xlsx")

--2026-06-25 14:09:04--  https://raw.githubusercontent.com/mzahidgurbuz/Turkish-Hate-Speech-Detection/main/9KTurkishHateSpeechDataset.xlsx
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2201327 (2.1M) [application/octet-stream]
Saving to: ‘9KTurkishHateSpeechDataset.xlsx’

9KTurkishHateSpeech 100%[===================>]   2.10M  --.-KB/s    in 0.04s   

2026-06-25 14:09:04 (54.4 MB/s) - ‘9KTurkishHateSpeechDataset.xlsx’ saved [2201327/2201327]



In [34]:
df2

,row_id,Tweet,Etiket1,Alt Etiket,Etiket2,Alt Etiket.1,Etiket,Alt Etiket.2,Tweet ID,Tweet Date,...,Username,User ID,User Description,User URL,User Language,User Location,User-Statuses,User-Followers,User-Friends,User-Favourites
0,1,@ZaferPartisi38 Sınırdan hergün binlerce kaçak...,nefret söylemi,Etnik,nefret söylemi,Etnik,nefret söylemi,Etnik,NaN,2022-04-03T23:59:40+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,13177,947,1337,86432
1,2,Len suriyeli ler gideceksiniz kardeşim lamı c...,nefret söylemi,Etnik,nefret söylemi,Etnik,nefret söylemi,Etnik,NaN,2022-04-03T23:58:04+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,12801,194,61,14503
2,3,@ajans_muhbir Sikseniz suriyeli mülteci sempat...,saldırgan,Etnik,nefret söylemi,Etnik,nefret söylemi,Etnik,NaN,2022-04-03T23:57:22+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,1965,16,226,2714
3,4,@MKA__ATAM Suriyeli oldukları ne malum?,hiçbiri,Etnik,hiçbiri,Etnik,hiçbiri,Etnik,NaN,2022-04-03T23:50:09+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,16418,558,1887,23125
4,5,Suriyeli konusunda iğneyi kendimize batırmak l...,hiçbiri,Etnik,hiçbiri,Etnik,hiçbiri,Etnik,NaN,2022-04-03T23:49:40+00:00,...,NaN,NaN,NaN,NaN,NaN,BEŞİKTAŞ,14993,11296,9737,40101
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9813,9996,NaN,hiçbiri,Cinsiyetçi,hiçbiri,Cinsiyetçi,hiçbiri,Cinsiyetçi,NaN,2022-03-27T11:50:08+00:00,...,colpanyildizi,9.760044e+17,Anne,https://twitter.com/colpanyildizi,NaN,NaN,3448,458,435,8581
9814,9997,NaN,hiçbiri,Cinsiyetçi,hiçbiri,Cinsiyetçi,hiçbiri,Cinsiyetçi,NaN,2022-03-27T11:40:40+00:00,...,atlassmom,1.143510e+18,NaN,https://twitter.com/atlassmom,https://pbs.twimg.com/profile_banners/11435104...,"Nilüfer, Türkiye",978,172,250,5347
9815,9998,NaN,hiçbiri,Cinsiyetçi,hiçbiri,Cinsiyetçi,hiçbiri,Cinsiyetçi,NaN,2022-03-27T11:29:26+00:00,...,HeviLgbt,1.618027e+09,Komeleya Hêvî JMNTN - Hevi LGBTİ+ Derneği - HE...,https://twitter.com/HeviLgbt,https://pbs.twimg.com/profile_banners/16180274...,Beyoğlu - Istanbul / TURKEY,7400,5913,396,3208
9816,9999,NaN,hiçbiri,Cinsiyetçi,hiçbiri,Cinsiyetçi,hiçbiri,Cinsiyetçi,NaN,2022-03-27T11:26:02+00:00,...,BAHARAMADOANIN,1.469001e+18,"Karım haricinde ki kimseye tahammülüm yok, irt...",https://twitter.com/BAHARAMADOANIN,https://pbs.twimg.com/profile_banners/14690010...,doğa,951,655,374,6484


In [35]:
df2=df2[["Tweet","Etiket1"]]

In [36]:
df2["Etiket1"].value_counts()

,count
Etiket1,
hiçbiri,5394
nefret söylemi,2785
saldırgan,1639


In [37]:
df2.loc[df2["Etiket1"] == "hiçbiri", "Etiket1"] = 0
df2.loc[df2["Etiket1"] == "nefret söylemi", "Etiket1"] = 1
df2.loc[df2["Etiket1"] == "saldırgan", "Etiket1"] = 1


/tmp/ipykernel_2330/1961660196.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2.loc[df2["Etiket1"] == "hiçbiri", "Etiket1"] = 0
/tmp/ipykernel_2330/1961660196.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2.loc[df2["Etiket1"] == "nefret söylemi", "Etiket1"] = 1
/tmp/ipykernel_2330/1961660196.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2.loc[df2["Etiket1"] == "saldırgan", "Etiket1"] = 1


In [38]:
df2["Etiket1"] = df2["Etiket1"].astype(int)

/tmp/ipykernel_2330/2894232081.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2["Etiket1"] = df2["Etiket1"].astype(int)


In [39]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9818 entries, 0 to 9817
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Tweet    301 non-null    object
 1   Etiket1  9818 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 153.5+ KB


In [40]:
df2["text_clean"]=df2["Tweet"].apply(metin_temizle)
print(df2[['Tweet', 'text_clean']])

                                                  Tweet  \
0     @ZaferPartisi38 Sınırdan hergün binlerce kaçak...   
1     Len suriyeli ler gideceksiniz kardeşim  lamı c...   
2     @ajans_muhbir Sikseniz suriyeli mülteci sempat...   
3               @MKA__ATAM Suriyeli oldukları ne malum?   
4     Suriyeli konusunda iğneyi kendimize batırmak l...   
...                                                 ...   
9813                                                NaN   
9814                                                NaN   
9815                                                NaN   
9816                                                NaN   
9817                                                NaN   

                                             text_clean  
0     sınırdan hergün binlerce kaçak giriyor ama ikt...  
1     len suriyeli ler gideceksiniz kardeşim lamı ci...  
2     sikseniz suriyeli mülteci sempatizanı olmayaca...  
3                           suriyeli oldukları ne malum  
4

/tmp/ipykernel_2330/4158183192.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2["text_clean"]=df2["Tweet"].apply(metin_temizle)


In [41]:
df2 = df2.dropna(subset=["Tweet"])

In [42]:
df2=df2[["Etiket1","text_clean"]]


In [43]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 301 entries, 0 to 300
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Etiket1     301 non-null    int64 
 1   text_clean  301 non-null    object
dtypes: int64(1), object(1)
memory usage: 7.1+ KB


In [44]:
new_df = pd.concat([df1, df2, df], ignore_index=True)

In [45]:
df1.info()
df2.info()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Etiket      1000 non-null   int64 
 1   text_clean  1000 non-null   object
dtypes: int64(1), object(1)
memory usage: 15.8+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 301 entries, 0 to 300
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Etiket1     301 non-null    int64 
 1   text_clean  301 non-null    object
dtypes: int64(1), object(1)
memory usage: 7.1+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41423 entries, 0 to 41422
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   text           41423 non-null  object 
 1   labels         41423 non-null  int64  
 2   source         41423 non-null  object 
 3   dataset        41423 non-null  object 
 4 

In [46]:
df=df[["text_clean","labels"]]

In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41423 entries, 0 to 41422
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   text_clean  41423 non-null  object
 1   labels      41423 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 647.4+ KB


In [48]:
df1.rename(columns={"Etiket": "labels"}, inplace=True)
df2.rename(columns={"Etiket1": "labels"}, inplace=True)

In [49]:
new_df = pd.concat([df1, df2, df], ignore_index=True)

In [50]:
new_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42724 entries, 0 to 42723
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   labels      42724 non-null  int64 
 1   text_clean  42724 non-null  object
dtypes: int64(1), object(1)
memory usage: 667.7+ KB


In [51]:
new_df["text_clean"].value_counts()

,count
text_clean,
,74
link,19
siktir git,15
orospu çocuğu,11
cevatdavulcu,8
...,...
cidden üzülüyorum bu kadına i̇nsanın geliştireceği ve bundan da gurur duyacağı bir beyni varken kadın sen git götünü geliştir onunla gurur duy millete de reklam et olmuyor hatice cidden zorlama artık,1
sfu1907amk bir de tik cekiyorlar ya günaydının önüne verem oluyorum amik,1
bir olerkan56 siktir git amk osu ekrana zarar veriyor sillik kazandıklarını bir yede otur amk,1


In [52]:
new_df["text_clean"] = new_df["text_clean"].str.strip()
new_df = new_df[new_df["text_clean"] != "cevatdavulcu"]

In [53]:
X=new_df["text_clean"]
y=new_df["labels"]

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model=LogisticRegression(max_iter=1000,class_weight="balanced")
model.fit(X_train_tfidf,y_train)

print(classification_report(y_test, model.predict(X_test_tfidf)))

              precision    recall  f1-score   support

           0       0.89      0.92      0.91      5736
           1       0.83      0.78      0.80      2808

    accuracy                           0.87      8544
   macro avg       0.86      0.85      0.85      8544
weighted avg       0.87      0.87      0.87      8544



In [54]:
X=df2["text_clean"]
y=df2["labels"]

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model=LogisticRegression(max_iter=1000,class_weight="balanced")
model.fit(X_train_tfidf,y_train)

print(classification_report(y_test, model.predict(X_test_tfidf)))

              precision    recall  f1-score   support

           0       0.61      0.46      0.52        24
           1       0.70      0.81      0.75        37

    accuracy                           0.67        61
   macro avg       0.65      0.63      0.64        61
weighted avg       0.66      0.67      0.66        61



**df1 ve df2 performansı düşürdüğü için direkt eliyoruz yanlızca df ile ilerleyeceğiz**


In [55]:
df


,text_clean,labels
0,iyi partili türkkan 15 milyon suriyeli daha ge...,0
1,başkan erdoğandan suriyeli mülteci mesajı ya d...,0
2,arslan bulut yazdı 300 metrekarelik konaklar y...,1
3,güzel bir amaçla yola çıkmış ama insan askerin...,0
4,kimin parası bu türk halkının parası peki bizi...,1
...,...,...
41418,nurayorganizasyon çok teşekkür ederim beğenmen...,0
41419,eminesunan12merhabalar arkadaşlar bu paylaşıml...,0
41420,iremmkrtls,0
41421,jila88j ay gülmekten karnima ağrilar gırdı sen...,0


In [56]:
df["labels"].value_counts()

,count
labels,
0,27903
1,13520


In [57]:
df.info()




<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41423 entries, 0 to 41422
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   text_clean  41423 non-null  object
 1   labels      41423 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 647.4+ KB


In [58]:
!pip install transformers datasets torch -q

In [59]:
from transformers import BertTokenizer ,BertForSequenceClassification,Trainer,TrainingArguments

In [60]:
from datasets import Dataset
import torch

In [61]:
from sklearn.model_selection import train_test_split
X=df["text_clean"]
y=df["labels"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=df["labels"])

In [62]:
X_train


,text_clean
2181,suriye halkına uygulanan dehşetlerin gerçekliğ...
17079,alimteke3025ahmetin yarrağı kalkmıyor muydu
8936,dolar ne olursa olsun butun gavur birleşsin el...
26914,türk_man1453 sen insan mısın mal kadın
40574,handanbutik35 haddini bildir vallahi
...,...
4369,i̇srail polisi korumasındaki 127 fanatik yahud...
19448,pusattt_17benim anlamadığım bu siktiğim yarışm...
28799,sedagul_arpaciselvii topuk ayrı da o dudak dol...
18209,ismailturna58derhal comolli ve çocuk yu kovun ...


In [63]:
X_test

,text_clean
19957,alisanofficial bokun tahtına bakan çisinde içm...
9246,1994te bu günde iki markale katliamından ilki ...
20683,ulan ne haysiyetsiz şeref yoksunu eğitimsiz ır...
25587,tugceorhan81 fena değil
6816,diger bir cok sorulan sey dil ve irkcilik yahu...
...,...
14374,perfectlenad ki̇mse çikip sila bağirip çağirdi...
15628,ananın amına kafamı sokayım ölmüşlerine bacak ...
30294,betulduzgunak kesin bilgi maalesef çok acı art...
29911,nermin cat pınar mertin hanımını çağırmamışlar...


## 5. BERT Fine-Tuning

In [64]:
train_df=pd.DataFrame({"text":X_train,"label":y_train})
test_df=pd.DataFrame({"text":X_test,"label":y_test})

train_dataset=Dataset.from_pandas(train_df)
test_dataset=Dataset.from_pandas(test_df)

In [65]:
tokenizer=BertTokenizer.from_pretrained("dbmdz/bert-base-turkish-cased")

In [66]:
def tokenize(batch):
  return tokenizer(batch["text"],padding="max_length",truncation=True,max_length=128)

In [67]:
train_dataset=train_dataset.map(tokenize,batched=True)
test_dataset=test_dataset.map(tokenize,batched=True)

Map:   0%|          | 0/33138 [00:00<?, ? examples/s]

Map:   0%|          | 0/8285 [00:00<?, ? examples/s]

In [68]:
model = BertForSequenceClassification.from_pretrained(
    'dbmdz/bert-base-turkish-cased',
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [69]:
train_dataset = train_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

In [70]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_dir='./logs',
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [71]:
!pip uninstall torchvision -y

In [72]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)
print(classification_report(y_test, preds))


## 6. Model Kaydetme

In [ ]:
trainer.save_model('./turkish_hate_model')
tokenizer.save_pretrained('./turkish_hate_model')
print("Kaydedildi!")

In [ ]:
import shutil
shutil.make_archive('turkish_hate_model', 'zip', './turkish_hate_model')
print("Zip hazır!")